# Đánh Giá Mô Hình Thị Giác - Ngôn Ngữ (MLLMs) Trên Benchmark RS-Solve (English)

Notebook này thực hiện suy luận và đo lường toàn diện các chỉ số khoa học theo kế hoạch nghiên cứu IEEE TGRS:
- **Các mô hình hỗ trợ:** Qwen2.5-VL-7B-Instruct, InternVL2.5-8B, LLaVA-1.5-7B, GeoChat-7B, và Closed APIs.
- **Chỉ số khoa học xuất bản:**
  - **BẢNG VI**: Ma trận kết cục $\rho_{px} \times \rho_{tok}$ (% Hallucination / % Lỗi nhận thức / % Từ chối)
  - **BẢNG VII**: Mô hình hồi quy logistic kiểm định giả thuyết nút thắt token $H_1$ (Hệ số $\beta_1$, CI 95%, p-value)
  - **BẢNG VIII**: Hai ngưỡng phân giải độc lập $H_2$ (Ngưỡng cảm biến $p_0^{50}, p_0^U$ và ngưỡng token $\rho_{tok}^*, \rho_{tok}^* \cdot P$)
  - **BẢNG IX**: Bậc thang Johnson $H_3$ & Kiểm định đơn điệu Jonckheere-Terpstra
  - **BẢNG X**: Dự đoán có chọn lọc Selective Prediction $H_4$ (Risk-Coverage, Cov@5%, Cov@10%, Cov@20%, AURC, Abstention-accuracy trên U)

In [ ]:
# [1] Cài đặt các thư viện cần thiết cho MLLM và tính toán chỉ số
!pip install -q "transformers>=4.49.0" "accelerate>=0.26.0" "qwen-vl-utils>=0.0.8" torchvision scipy pillow

import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Dung lượng : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# [2] Tải mã nguồn mới nhất từ GitHub
import os
import sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working/rgsp_nckh")
if not WORK_DIR.exists():
    !git clone https://github.com/kttt294/RGSP-nckh.git {WORK_DIR}
else:
    !cd {WORK_DIR} && git pull origin main

sys.path.insert(0, str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR / "rs-solve"))
sys.path.insert(0, str(WORK_DIR / "rs-solve" / "eval"))
print(f"Thư mục làm việc: {WORK_DIR}")

In [ ]:
# [3] Cấu hình tham số đánh giá
# Các lựa chọn mô hình: 'qwen2.5-vl', 'internvl', 'llava-1.5', 'geochat', 'mock'
SELECTED_MODEL = "qwen2.5-vl"

# Số lượng mẫu tối đa cần đánh giá:
# - Đặt None để chạy TOÀN BỘ 30,771 câu hỏi benchmark
# - Đặt số nguyên (ví dụ 1000) để chạy pilot test nhanh
MAX_SAMPLES = None

# Bật lấy mẫu phân tầng cân bằng các loại câu hỏi nếu chạy mẫu con
STRATIFIED = True

BENCHMARK_FILE = WORK_DIR / "kaggle_benchmark_output" / "rs_solve_full_benchmark_en.jsonl"
IMAGE_ROOT = Path("/kaggle/input")
OUTPUT_PREDS = Path(f"/kaggle/working/eval_outputs/preds_{SELECTED_MODEL.replace('-', '_').replace('.', '_')}.jsonl")
OUTPUT_REPORTS = Path("/kaggle/working/eval_reports")

OUTPUT_PREDS.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_REPORTS.mkdir(parents=True, exist_ok=True)

print(f"Mô hình được chọn : {SELECTED_MODEL}")
print(f"File Benchmark   : {BENCHMARK_FILE}")
print(f"File xuất dự đoán: {OUTPUT_PREDS}")

In [ ]:
# [4] Khởi chạy quá trình suy luận (Inference)
from run_inference import get_model_adapter, run_evaluation

device = "cuda" if torch.cuda.is_available() else "cpu"
adapter = get_model_adapter(SELECTED_MODEL, device=device)

run_evaluation(
    model_adapter=adapter,
    benchmark_file=BENCHMARK_FILE,
    image_root=IMAGE_ROOT,
    output_file=OUTPUT_PREDS,
    max_samples=MAX_SAMPLES,
    resume=True,
    stratified=STRATIFIED
)

In [ ]:
# [5] Tính toán toàn bộ các bảng chỉ số IEEE TGRS (Tables VI, VII, VIII, IX, X)
from compute_metrics import evaluate_prediction_file
from IPython.display import Markdown, display

P_PATCH = 32 if "qwen" in SELECTED_MODEL else (28 if "internvl" in SELECTED_MODEL else 14)
results = evaluate_prediction_file(OUTPUT_PREDS, OUTPUT_REPORTS, p_patch=P_PATCH)

# Hiển thị báo cáo Markdown trực quan
md_report_file = OUTPUT_REPORTS / f"eval_report_{OUTPUT_PREDS.stem}.md"
if md_report_file.exists():
    display(Markdown(md_report_file.read_text(encoding="utf-8")))

In [ ]:
# [6] Nén kết quả để tải về lưu trữ
!tar -czvf /kaggle/working/eval_results_{SELECTED_MODEL}.tar.gz /kaggle/working/eval_outputs /kaggle/working/eval_reports
print("Hoàn tất nén kết quả tại /kaggle/working/eval_results_{SELECTED_MODEL}.tar.gz")